# LAB : Spark MLlib

### Étape 0 : Initialisation de Spark

In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

spark = SparkSession.builder.appName("Lab4").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

### Étape 1 : Stocker le contenu de tous les fichiers du dossier tp4_data dans un DataFrame

In [ ]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("C:/Users/User/Desktop/tp4_Spark_Ml/tp4_data/tp4_data")

df.show(5)

### Étape 2 : Afficher le schéma du DataFrame obtenu

In [ ]:
df.printSchema()

### Étape 3 : Remplir les valeurs manquantes (NaN) avec la valeur 0

In [ ]:
df = df.na.fill(0)

### Étape 4 : Ajouter une nouvelle colonne "day_of_week" (jour de la semaine correspondant à "InvoiceDate")

In [ ]:
df = df.withColumn("day_of_week", F.date_format(F.col("InvoiceDate"), "EEEE"))

df.select("InvoiceDate", "day_of_week").show(5)

### Étape 5 : Diviser les données en ensemble d'apprentissage (avant 2010-12-13) et ensemble de test (à partir de 2010-12-13)

In [ ]:
train = df.where("InvoiceDate < '2010-12-13'")
test = df.where("InvoiceDate >= '2010-12-13'")

print(train.count(), test.count())

### Étape 6 : Créer un StringIndexer pour transformer les jours de la semaine en valeurs numériques

In [ ]:
indexer = StringIndexer() \
    .setInputCol("day_of_week") \
    .setOutputCol("day_of_week_index") \
    .setHandleInvalid("keep")

### Étape 7 : Éviter l'ordre implicite entre les jours (samedi > lundi) : comment résoudre ce problème ?

In [ ]:
encoder = OneHotEncoder() \
    .setInputCol("day_of_week_index") \
    .setOutputCol("day_of_week_encoded") 

# avec one hot encoder le probleme de 6 > 1 laisee croire que samdi est plus grand que lundi , mais avec one hot chaque jour deviant un vecteur ou un seule case vaut 1 et lesa autres vont a zero 

### Étape 8 : Créer un VectorAssembler avec UnitPrice, Quantity et day_of_week_encoded

In [ ]:
vectorAssembler = VectorAssembler() \
    .setInputCols(["UnitPrice", "Quantity", "day_of_week_encoded"]) \
    .setOutputCol("features")

### Étape 9 : Créer un pipeline configuré avec les résultats des étapes 6, 7 et 8

In [ ]:
pipeline = Pipeline().setStages([indexer, encoder, vectorAssembler])

### Étape 10 : Le StringIndexer doit connaître le nombre de valeurs uniques à indexer : comment résoudre ce problème ?

In [ ]:
fittedPipeline = pipeline.fit(train)

### Étape 11 : Transformer les données de l'ensemble d'apprentissage en se basant sur les stages du pipeline

In [ ]:
transformedTraining = fittedPipeline.transform(train)

transformedTraining.select("UnitPrice", "Quantity", "day_of_week", "day_of_week_index", "day_of_week_encoded", "features").show(5, truncate=False)

### Étape 12 : Créer une instance de KMeans avec 20 clusters

In [ ]:
kmeans = KMeans().setK(20).setSeed(1).setFeaturesCol("features")

### Étape 13 : Lancer l'apprentissage de KMeans sur l'ensemble obtenu à l'étape 11

In [ ]:
kmModel = kmeans.fit(transformedTraining)

### Étape 14 : Effectuer des prédictions sur l'ensemble de test

In [ ]:
transformedTest = fittedPipeline.transform(test)
predictions = kmModel.transform(transformedTest)

predictions.select("UnitPrice", "Quantity", "day_of_week", "prediction").show(10)

### Étape 15 : Calculer le coefficient de silhouette

In [ ]:
evaluator = ClusteringEvaluator() \
    .setPredictionCol("prediction") \
    .setFeaturesCol("features") \
    .setMetricName("silhouette")

silhouette = evaluator.evaluate(predictions)
print(silhouette)